In [1]:
import mediapipe as mp
import numpy as np
mp_pose = mp.solutions.pose
import cv2
import os,socket,sys,time
import spidev as SPI
import xgoscreen.LCD_2inch as LCD_2inch
from PIL import Image,ImageDraw,ImageFont
from key import Button
from xgolib import XGO
from picamera2 import Picamera2

dog = XGO(port='/dev/ttyAMA0',version="xgolite")

In [2]:
# 导入组件 Importing Components
import ipywidgets.widgets as widgets
image_widget = widgets.Image(format='jpeg', width=320, height=240)  #设置摄像头显示组件  Set up the camera display component

# 将BGR图像转换为JPEG格式的字节流 Convert a BGR image to a JPEG byte stream
def bgr8_to_jpeg(value, quality=75):
    return bytes(cv2.imencode('.jpg', value)[1])

#display(image_widget) #显示出来

In [3]:
mydisplay = LCD_2inch.LCD_2inch()
mydisplay.clear()
splash = Image.new("RGB", (mydisplay.height, mydisplay.width ),"black")
mydisplay.ShowImage(splash)
button=Button()
#-----------------------COMMON INIT-----------------------
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_holistic = mp.solutions.holistic
joint_list = [[24,26,28], [23,25,27], [14,12,24], [13,11,23]]  # leg&arm

# For webcam input:
picam2 = Picamera2()
picam2.configure(
    picam2.create_preview_configuration(main={"format": "RGB888", "size": (320, 240)})
)
picam2.start()

[0:15:06.037864942] [257407]  INFO Camera camera_manager.cpp:325 libcamera v0.3.2+99-1230f78d
[0:15:06.057987141] [257528]  INFO RPI pisp.cpp:695 libpisp version v1.0.7 28196ed6edcf 29-08-2024 (16:33:32)
[0:15:06.090284829] [257528]  INFO RPI pisp.cpp:1154 Registered camera /base/axi/pcie@120000/rp1/i2c@88000/ov5647@36 to CFE device /dev/media2 and ISP device /dev/media0 using PiSP variant BCM2712_D0
[0:15:06.094344813] [257407]  INFO Camera camera.cpp:1197 configuring streams: (0) 320x240-RGB888 (1) 640x480-GBRG_PISP_COMP1
[0:15:06.094443906] [257528]  INFO RPI pisp.cpp:1450 Sensor: /base/axi/pcie@120000/rp1/i2c@88000/ov5647@36 - Selected sensor format: 640x480-SGBRG10_1X10 - Selected CFE format: 640x480-PC1g


In [4]:
display(image_widget) #显示出来
with mp_pose.Pose(
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5) as pose:
  while True:
    frame = picam2.capture_array() 
    image = cv2.flip(frame, 1)


    # To improve performance, optionally mark the image as not writeable to
    # pass by reference.
    image.flags.writeable = False
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = pose.process(image)

    # Draw the pose annotation on the image.
    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    mp_drawing.draw_landmarks(
        image,
        results.pose_landmarks,
        mp_pose.POSE_CONNECTIONS,
        landmark_drawing_spec=mp_drawing_styles.get_default_pose_landmarks_style())
    # Flip the image horizontally for a selfie-view display.
    
    if results.pose_landmarks:
        RHL = results.pose_landmarks
        angellist=[]
        for joint in joint_list:
            a = np.array([RHL.landmark[joint[0]].x, RHL.landmark[joint[0]].y])
            b = np.array([RHL.landmark[joint[1]].x, RHL.landmark[joint[1]].y])
            c = np.array([RHL.landmark[joint[2]].x, RHL.landmark[joint[2]].y])
            radians_fingers = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
            angle = np.abs(radians_fingers * 180.0 / np.pi) 
            if angle > 180.0:
                angle = 360 - angle
            #cv2.putText(image, str(round(angle, 2)), tuple(np.multiply(b, [640, 480]).astype(int)),cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2, cv2.LINE_AA)
            angellist.append(angle)
    else:
        angellist=[]
        dog.reset()
    print(angellist)
    b,g,r = cv2.split(image)
    image = cv2.merge((r,g,b))
    image = cv2.flip(image, 1)
    try:
        res=str(int(angellist[0]))+'|'+str(int(angellist[1]))+'|'+str(int(angellist[2]))+'|'+str(int(angellist[3]))
        #dog.leg(1,[0,-18+int(angellist[2]/10),0])
        #dog.leg(2,[0,-18+int(angellist[3]/10),0])
    except:
        res=' '
    cv2.putText(image,res,(10,220),cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2, cv2.LINE_AA)
    imgok = Image.fromarray(image)
    mydisplay.ShowImage(imgok)

    r,g,b = cv2.split(image)
    imagecv2 = cv2.merge((b,g,r))
    image_widget.value = bgr8_to_jpeg(imagecv2)
    
    # cv2.imshow('MediaPipe Pose', imagecv2)

    if cv2.waitKey(5) & 0xFF == 27:
        break
    if button.press_b():
        dog.reset()
        break

picam2.stop()
picam2.close()
del dog
print('done!')

Image(value=b'', format='jpeg', height='240', width='320')

Error in cpuinfo: prctl(PR_SVE_GET_VL) failed
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1758522325.082854  257534 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1758522325.161523  257534 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1758522325.234052  257534 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


[179.88869036808111, 147.76575714781262, 33.630686178943805, 42.90132923585302]
[170.27490267131478, 137.42974308310843, 30.704109918383256, 36.835143705186994]
[167.88999610417315, 138.381981248497, 28.854624754276315, 32.711239858745856]
[158.38438217549557, 128.01472436977258, 28.09050724705142, 30.04267677814845]
[139.88203889521697, 116.49049455006993, 27.0818628418134, 26.709609387351026]
[159.02636977548173, 131.0299411411992, 26.88304651260999, 27.07290158788865]
[167.43004008188595, 143.644610429077, 27.137375260282656, 27.809644314740545]
[169.29904520985892, 148.7238697363898, 27.09298547664197, 27.605780512049577]
[170.3692312182438, 150.26500970132722, 24.480790734091002, 24.333686828063794]
[173.98783878171562, 156.8736458339202, 21.81508782559204, 22.587572321455024]
[172.41427542905737, 145.9907178849539, 23.55255316881713, 23.606197594387638]
[172.40543635884677, 149.33286974659572, 24.851391239671408, 26.701537872109345]
[173.344484497725, 149.70923635896995, 25.89131

KeyboardInterrupt: 

In [ ]:
picam2.stop()
picam2.close()
del dog